In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals

# standard python
import numpy as np
import pickle 
import deepchem as dc
import scipy
from scipy import stats
import os
import matplotlib
from matplotlib import pyplot as plt
from IPython.display import Image
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from scipy.spatial.distance import pdist
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Layer, LayerNormalization, Dense, Input, Reshape
tf.keras.backend.clear_session()

In [ ]:
RDLogger.DisableLog('rdApp.*')

SEED = 42
MIN_DIST = 0.5

bbbp = pd.read_csv("BBBP.csv")

def largest_fragment(mol):
    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
    return mol if len(frags) == 1 else max(frags, key=lambda m: m.GetNumHeavyAtoms())

params = AllChem.ETKDGv3()
params.randomSeed = SEED
params.useRandomCoords = True
params.maxAttempts = 10

mollist, labels, smiles_list = [], [], []
drops = {"parse": 0, "embed": 0, "uff_params": 0, "not_converged": 0, "overlap": 0}

for _, row in bbbp.iterrows():
    mol = Chem.MolFromSmiles(row["smiles"])
    if mol is None:
        drops["parse"] += 1; continue

    mol = largest_fragment(mol)
    canon = Chem.MolToSmiles(mol)
    mol = Chem.AddHs(mol)

    if AllChem.EmbedMolecule(mol, params) != 0:
        drops["embed"] += 1; continue
    if not AllChem.UFFHasAllMoleculeParams(mol):
        drops["uff_params"] += 1; continue
    if AllChem.UFFOptimizeMolecule(mol, maxIters=2000) != 0:
        drops["not_converged"] += 1; continue

    coords = mol.GetConformer().GetPositions().astype(np.float32)
    if pdist(coords).min() < MIN_DIST:
        drops["overlap"] += 1; continue

    mollist.append(list(zip([a.GetSymbol() for a in mol.GetAtoms()], coords)))
    labels.append(row["p_np"])
    smiles_list.append(canon)

y = np.array(labels, dtype=np.int64).reshape(-1, 1)

print(f"kept {len(mollist)} of {len(bbbp)}")
print("dropped:", drops)
print(f"positive rate: {y.mean():.4f}")

with open("bbbp_3d.pkl", "wb") as f:
    pickle.dump({"mollist": mollist, "y": y, "smiles": smiles_list, "seed": SEED},
                f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
def speciesmap(atom_type):
    atom_to_number = {
        'H': 1,
        'B': 5,
        'C': 6,
        'N': 7,
        'O': 8,
        'F': 9,
        'Si': 14,
        'P': 15,
        'S': 16,
        'Cl': 17,
        'Se': 34,
        'Br': 35,
        'I': 53
    }

    return np.array([atom_to_number.get(atom_type, 0)])

In [ ]:
from bbbp_weightedviews import load_bbbp_data
ws, vs, Natoms, Nviews = load_bbbp_data(mollist, speciesmap, setNatoms=None,
    setNviews=None,
    carbonbased=False,
    not_hydrogen=False,
    heaviest_origin=False,
    verbose=0
)
print("ws shape:", ws.shape)
print("vs shape:", vs.shape)
print("Natoms:", Natoms)
print("Nviews:", Nviews)

In [ ]:

train_mollist, test_mollist, y_train, y_test = train_test_split(
    mollist,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y
)


bbbp_labels_train = y_train
bbbp_labels_test = y_test




In [ ]:

ws_bbbp_train, vs_bbbp_train, Natoms_train, Nviews_train = load_bbbp_data(
    train_mollist,
    speciesmap,
    setNatoms=None,
    setNviews=None,
    carbonbased=False,
    not_hydrogen=False,
    heaviest_origin=False,
    verbose=0
)

bbbpG_train = [ws_bbbp_train, vs_bbbp_train]


ws_bbbp_test, vs_bbbp_test, Natoms_test, Nviews_test = load_bbbp_data(
    test_mollist,
    speciesmap,
    setNatoms=None,
    setNviews=None,
    carbonbased=False,
    not_hydrogen=False,
    heaviest_origin=False,
    verbose=0
)

bbbpG_test = [ws_bbbp_test, vs_bbbp_test]

In [ ]:
from sklearn.preprocessing import StandardScaler

# Make copies
Xtr = atom_embeddings_bbbp_train.copy()
Xte = atom_embeddings_bbbp_test.copy()

# Identify real atoms (exclude zero-padded atoms)
mask_tr = (Xtr[..., 0] != 0)
mask_te = (Xte[..., 0] != 0)

# Fit scaler ONLY on training atomic properties
props_tr = Xtr[..., 4:12][mask_tr]

prop_scaler = StandardScaler().fit(props_tr)

# Scale train and test using training statistics
Xtr[..., 4:12][mask_tr] = prop_scaler.transform(
    Xtr[..., 4:12][mask_tr]
)

Xte[..., 4:12][mask_te] = prop_scaler.transform(
    Xte[..., 4:12][mask_te]
)

print("Xtr:", Xtr.shape)
print("Xte:", Xte.shape)

In [ ]:
from bbbp_single_atom import (
    small_views,
    get_embeddings,
    atom_properties,
    single_atomic_property_switches
)

# Split views into individual atoms 
single_atom_bbbp_train = small_views(vs_bbbp_train)
single_atom_bbbp_test = small_views(vs_bbbp_test)

# Generate atomic embeddings
atom_embeddings_bbbp_train = get_embeddings(
    single_atom_bbbp_train,
    atom_properties,
    single_atomic_property_switches
)

atom_embeddings_bbbp_test = get_embeddings(
    single_atom_bbbp_test,
    atom_properties,
    single_atomic_property_switches
)

print("Train embeddings:", atom_embeddings_bbbp_train.shape)
print("Test embeddings:", atom_embeddings_bbbp_test.shape)

In [ ]:
labelsG_train = bbbp_labels_train
labelsG_test = bbbp_labels_test

Ntoxicity = 1

ws_train, vs_train = ws_bbbp_train, Xtr
ws_test, vs_test = ws_bbbp_test, Xte

dataG_train = [ws_train, vs_train]
dataG_test = [ws_test, vs_test]

In [ ]:
def compute_coulomb_matrix_for_view(view):
    """

    """
    n_atoms = view.shape[0]
    coulomb_matrix = np.zeros((n_atoms, n_atoms))
    
    for i in range(n_atoms):
        atomic_num_i = view[i, 0]
        if atomic_num_i == 0:
            continue
            
        coords_i = view[i, 1:4]
        
        for j in range(n_atoms):
            atomic_num_j = view[j, 0]
            if atomic_num_j == 0:  # Skip padding atoms
                continue
                
            coords_j = view[j, 1:4]
            
            if i == j:
                # Diagonal element: self-interaction
                coulomb_matrix[i, j] = 0.5 * atomic_num_i ** 2.4
            else:
                # Off-diagonal element: interaction between atoms i and j
                dist = np.linalg.norm(coords_i - coords_j)
                coulomb_matrix[i, j] = atomic_num_i * atomic_num_j / dist
                
    return coulomb_matrix

In [ ]:
def precompute_coulomb_matrices_for_views(views_data):
    """
    Precompute Coulomb matrices for all views in the dataset.
    
    Parameters:
    - views_data: numpy array of shape (batch_size, num_views, num_atoms, 4)
    
    Returns:
    - coulomb_matrices: numpy array of shape (batch_size, num_views, num_atoms, num_atoms)
    """
    batch_size, num_views, num_atoms, _ = views_data.shape
    coulomb_matrices = np.zeros((batch_size, num_views, num_atoms, num_atoms))
    
    for batch_idx in range(batch_size):
        for view_idx in range(num_views):
            view = views_data[batch_idx, view_idx]
            coulomb_matrices[batch_idx, view_idx] = compute_coulomb_matrix_for_view(view)
                
    return coulomb_matrices

In [ ]:
coulomb_train = precompute_coulomb_matrices_for_views(
    single_atom_bbbp_train
)

coulomb_test = precompute_coulomb_matrices_for_views(
    single_atom_bbbp_test
)

print("Coulomb train:", coulomb_train.shape)
print("Coulomb test:", coulomb_test.shape)

In [ ]:

class TwoHeadAtomAttention(Layer):
    def __init__(self, d_k=32, d_v=32, **kwargs):
        super(TwoHeadAtomAttention, self).__init__(**kwargs)
        self.d_k = d_k
        self.d_v = d_v
        self.norm = LayerNormalization(axis=-1, epsilon=1e-6)

    def build(self, input_shape):
        self.num_views = input_shape[1]
        self.num_atoms = input_shape[2]
        self.feature_dim = input_shape[3]

        # ===== HEAD 1 =====
        self.W_q1 = self.add_weight(shape=(self.feature_dim, self.d_k), initializer='glorot_uniform', trainable=True)
        self.W_k1 = self.add_weight(shape=(self.feature_dim, self.d_k), initializer='glorot_uniform', trainable=True)
        self.W_v1 = self.add_weight(shape=(self.feature_dim, self.d_v), initializer='glorot_uniform', trainable=True)

        # ===== HEAD 2 =====
        self.W_q2 = self.add_weight(shape=(self.feature_dim, self.d_k), initializer='glorot_uniform', trainable=True)
        self.W_k2 = self.add_weight(shape=(self.feature_dim, self.d_k), initializer='glorot_uniform', trainable=True)
        self.W_v2 = self.add_weight(shape=(self.feature_dim, self.d_v), initializer='glorot_uniform', trainable=True)

        # Final projection
        self.W_o = self.add_weight(shape=(2 * self.d_v, self.feature_dim),
                                  initializer='glorot_uniform', trainable=True)

        # Coulomb scaling
        self.alpha = self.add_weight(shape=(), initializer=tf.keras.initializers.Constant(0.1), trainable=True)

    def attention_head(self, x, W_q, W_k, W_v, coulomb_matrix):
        Q = tf.matmul(x, W_q)
        K = tf.matmul(x, W_k)
        V = tf.matmul(x, W_v)

        scores = tf.matmul(Q, K, transpose_b=True)
        scores = scores / tf.math.sqrt(tf.cast(self.d_k, tf.float32))

        
        row_sums_before = tf.reduce_sum(scores, axis=2)
        zero_rows = tf.equal(row_sums_before, 0.0)

        # Add Coulomb
        if coulomb_matrix is not None:
            scores += self.alpha * coulomb_matrix

        # Softmax
        attn = tf.nn.softmax(scores, axis=-1)

        
        zero_rows = tf.expand_dims(zero_rows, axis=-1)
        attn = tf.where(zero_rows, tf.zeros_like(attn), attn)

        output = tf.matmul(attn, V)
        return output

    def call(self, inputs, **kwargs):
        coulomb_matrix = kwargs.get('coulomb_matrix', None)

        batch_size = tf.shape(inputs)[0]

        # Reshape
        x = tf.reshape(inputs, [-1, self.num_atoms, self.feature_dim])

        if coulomb_matrix is not None:
            coulomb_matrix = tf.reshape(coulomb_matrix, [-1, self.num_atoms, self.num_atoms])

        # ===== TWO HEADS =====
        out1 = self.attention_head(x, self.W_q1, self.W_k1, self.W_v1, coulomb_matrix)
        out2 = self.attention_head(x, self.W_q2, self.W_k2, self.W_v2, coulomb_matrix)

        # Concatenate
        concat = tf.concat([out1, out2], axis=-1) 

        # Project back
        projected = tf.matmul(concat, self.W_o)  

        # Add & Norm
        out = self.norm(projected + x)

        # Reshape back
        out = tf.reshape(out, [batch_size, self.num_views, self.num_atoms, self.feature_dim])

        return out


class FeedForward(Layer):
    def __init__(self, d_model=16, d_ff=128, **kwargs):
        super(FeedForward, self).__init__(**kwargs)
        self.fc1 = Dense(d_ff, activation='relu')
        self.fc2 = Dense(d_model)
        self.norm = LayerNormalization(axis=-1, epsilon=1e-6)

    def call(self, x):
        y = self.fc1(x)
        y = self.fc2(y)
        return self.norm(x + y)


class TransformerBlock(Layer):
    def __init__(self, d_model=16, d_k=32, d_v=32, d_ff=128, **kwargs):
        super(TransformerBlock, self).__init__(**kwargs)
        self.attn = TwoHeadAtomAttention(d_k=d_k, d_v=d_v)
        self.ffn = FeedForward(d_model=d_model, d_ff=d_ff)

    def call(self, inputs, **kwargs):
        coulomb_matrix = kwargs.get('coulomb_matrix', None)

        x = self.attn(inputs, coulomb_matrix=coulomb_matrix)  # Add & Norm 1
        x = self.ffn(x)                                       # Add & Norm 2

        return x

In [ ]:
# generic dense NN
def multiDense(Nin,Nout,Nhidden,widthhidden=None,kernel_regularizer=None):
    """Construct a basic NN with some dense layers.
    
    :parameter Nin: The number of inputs
    :type Nin: int
    :parameter Nout: The number of outputs
    :type Nout: int
    :parameter Nhidden: The number of hidden layers.
    :type Nhidden: int
    :parameter widthhidden: The width of each hidden layer.
        If left at None, Nin + Nout will be used.
    :parameter kernel_regularizer: the regularizer to use, such as regularizers.l2(0.001)
    :type kernel_regularizer: tensorflow.keras.regularizers.xxx
    :returns: The NN model
    :rtype: keras.Model
    
    """
    if widthhidden is None:
        widthhidden = Nin + Nout
    x = inputs = keras.Input(shape=(Nin,), name='multiDense_input')
    if kernel_regularizer is not None:
        print("Using regularization")
    for i in range(Nhidden):
        x = layers.Dense(widthhidden, activation='relu', kernel_regularizer=kernel_regularizer,name='dense'+str(i))(x)

    outputs = layers.Dense(Nout,name='multiDense_output')(x)
    return keras.Model(inputs=inputs, outputs=outputs)


In [ ]:
def parallelwrapper(Nparallel,basemodel,insteadmax=False):
    """Construct a model that applies a basemodel multiple times and take a weighted sum (or max) of the result.
    
    :parameter Nparallel: The number of times to apply in parallel
    :type Nparallel: int
    :parameter basemodel: a keras.Model inferred to have Nin inputs and Nout outputs.
    :type basemodel: a keras.Model
    :parameter insteadmax: If True, take the max of the results of the basemodel instead of the weighted sum.
        For compatibility, the model is still constructed with weights as inputs, but it ignores them.
    :type insteadmax: Boolean
    :returns: model with inputs shape [(?,Nparallel),(?,Nin,Nparallel)] and outputs shape (?,Nout).
        The first input is the scalar weights in the sum.
    :rtype: keras.Model
    
    Note: We could do a max over the parallel applications instead of or in addition to the weighted sum.
    
    """
    # infer shape of basemodel inputs and outputs
    Nin =  basemodel.inputs[0].shape[1]
    Nout =  basemodel.outputs[0].shape[1]
    
    # Apply basemodel Nparallel times in parallel
    # create main input (?,Nparallel,Nin) 
    parallel_inputs = keras.Input(shape=(Nparallel,Nin), name='parallelwrapper_input0')
    # apply base NN to each parallel slice; outputs (?,Nparallel,Nout)
    if False:
        # original version, stopped working at some tensorflow update
        xb = basemodel(parallel_inputs) # worked in earlier tensorflow
        #xb = tf.map_fn(basemodel,parallel_inputs) # another version that fails
    else:
        # newer version, works but makes summary and graphing cumbersome
        # unstack in the Nparallel directio
        parallel_inputsunstacked = tf.keras.ops.unstack(parallel_inputs, Nparallel, 1)
        # apply base NN to each 
        xbunstacked = [basemodel(x) for x in parallel_inputsunstacked]
        # re-stack
        xb = tf.keras.ops.stack(xbunstacked,axis=1)
    
    # create input scalars for weighted sun (?,Nparallel)
    weight_inputs = keras.Input(shape=(Nparallel,), name='parallelScalars')
    if insteadmax:
        # take max over the Nparallel direction to get (?,1,Nout)
        out = layers.MaxPool1D(pool_size=Nparallel)(xb)
        # reshape to (?,Nout)
        out = layers.Reshape((Nout,))(out)
    else:
        # do a weighted sum over the Nparallel direction to get (?,Nout)
        out = layers.Dot((-2,-1))([xb,weight_inputs])
    
    return keras.Model(inputs=[weight_inputs,parallel_inputs], outputs=out, name='parallelwrapper')

  
   

In [ ]:
def init_generator(data, labels, coulomb_matrices,
                   baselayers, Nfeatures, endlayers,
                   base_regularizer=None, end_regularizer=None,
                   num_blocks=3, d_ff=128):

    # 1) Extract dims
    Nviews = data[0].shape[1]   
    Natoms = data[1].shape[2]   
    d_model = 16        
    flattened_dim = Natoms * d_model 

    # 2) Inputs
    weight_input = Input((Nviews,), name='weight_input')             
    atom_input = Input((Nviews, Natoms, d_model), name='atom_input') 
    coulomb_input = Input((Nviews, Natoms, Natoms), name='coulomb_input') 

    # 3) Stacked Transformer blocks
    x = atom_input
    for i in range(num_blocks):
        x = TransformerBlock(
                d_model=d_model,
                d_k=32,
                d_v=32,
                d_ff=d_ff,
                name=f'transformer_block_{i}'
            )(x, coulomb_matrix=coulomb_input)

    # 4) Flatten & pool
    x = Reshape((Nviews, flattened_dim), name='flatten_views')(x) 

    Gbase = multiDense(flattened_dim, Nfeatures, baselayers,
                       kernel_regularizer=base_regularizer)
    
    Gpw = parallelwrapper(Nviews, Gbase, insteadmax=False)
    x = Gpw([weight_input, x])  # (batch,Nfeatures)

    # 5) Final MLP 
    Gft = multiDense(Nfeatures, 1, endlayers,
                     kernel_regularizer=end_regularizer)
    output = Gft(x)  # (batch,1)
    output = layers.Activation('sigmoid', name='sigmoid_output')(output)

    # 6) Assemble & compile
    generator = Model([weight_input, atom_input, coulomb_input], output,
                      name='generator_with_coulomb_attention')
    generator.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return generator


# ===== DATA PREPARATION =====

dataG_train = [
    np.array(dataG_train[0], dtype='float32'),
    np.array(dataG_train[1], dtype='float32'),
    np.array(coulomb_train, dtype='float32')
]

labelsG_train = np.array(labelsG_train, dtype='float32').reshape(-1, 1)

dataG_test = [
    np.array(dataG_test[0], dtype='float32'),
    np.array(dataG_test[1], dtype='float32'),
    np.array(coulomb_test, dtype='float32')
]

labelsG_test = np.array(labelsG_test, dtype='float32').reshape(-1, 1)


# ===== HYPERPARAMETERS =====

baselayers = 4
base_reg = 0
Nfeatures = 15
endlayers = 4
end_reg = 0

base_regularizer = regularizers.l2(base_reg) if base_reg else None
end_regularizer = regularizers.l2(end_reg) if end_reg else None


# ===== BUILD MODEL =====

generator = init_generator(
    dataG_train, labelsG_train, coulomb_train,
    baselayers, Nfeatures, endlayers,
    base_regularizer=base_regularizer,
    end_regularizer=end_regularizer
)

In [ ]:
%%time

BATCH_SIZE = 8

from tensorflow.keras.optimizers import AdamW
optimizer = AdamW(learning_rate=0.001)
generator.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=100,
    restore_best_weights=True,
    verbose=0
)

if 1:
    history = generator.fit(
        dataG_train,
        labelsG_train,
        epochs=200,
        batch_size=BATCH_SIZE,
        validation_data=(dataG_test, labelsG_test),
        callbacks=[early_stop],
        verbose=1
    )
    
    print("Train evaluation:")
    train_loss, train_acc = generator.evaluate(dataG_train, labelsG_train, verbose=1)
    print(f"Train loss = {train_loss:.4f}, Train accuracy = {train_acc:.4f}")
    
    print("Test evaluation:")
    test_loss, test_acc = generator.evaluate(dataG_test, labelsG_test, verbose=1)
    print(f"Test loss = {test_loss:.4f}, Test accuracy = {test_acc:.4f}")

In [ ]:
from sklearn.metrics import roc_auc_score

# Make predictions
train_pred = generator.predict(dataG_train)
test_pred = generator.predict(dataG_test)

# Calculate AUC-ROC
train_auc = roc_auc_score(labelsG_train, train_pred)
test_auc = roc_auc_score(labelsG_test, test_pred)

print(f"Train AUC-ROC = {train_auc:.4f}")
print(f"Test AUC-ROC = {test_auc:.4f}")